In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("yolov8n.pt")
model.predict(
    source="/content/drive/MyDrive/Traffic Dataset/images/test",
    show=True,
    save=True,
    classes=[2,3,5,7],   # vehicle only
    conf=0.25
)

WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/278 /content/drive/MyDrive/Traffic Dataset/images/test/00 (1).png: 640x608 1 car, 466.0ms
image 2/278 /content/drive/MyDrive/Traffic Dataset/images/test/00 (10).png: 640x640 3 cars, 1 motorcycle, 496.9ms
image 3/278 /content/drive/MyDrive/Traffic Dataset/images/test/00 (100).png: 384x640 3 cars, 1 motorcycle, 1 bus, 1 truck, 414.3ms
image 4/278 /content/drive/MyDrive/Traffi

In [ ]:
# Vehicle count from percentages (example: car 20%, bus 50%)
# Edit `total_vehicles` and the `percentages` dict for your case.

def counts_from_percentages(total_vehicles: int, percentages: dict, other_label: str = "other") -> dict:
    total_pct = sum(percentages.values())
    if total_pct > 100 + 1e-9:
        raise ValueError(f"Percentages sum to {total_pct}%, but must be <= 100%")

    # Initial rounding to integers
    counts = {k: int(round(total_vehicles * (p / 100.0))) for k, p in percentages.items()}

    # Fix any rounding difference by adding it to the largest class
    diff = total_vehicles - sum(counts.values())
    if diff != 0:
        largest_key = max(percentages, key=percentages.get)
        counts[largest_key] += diff

    # If user didn't specify 100%, put the remainder into `other`
    remainder_pct = 100.0 - total_pct
    if remainder_pct > 1e-9:
        used = sum(counts.values())
        counts[other_label] = total_vehicles - used

    return counts


total_vehicles = 100
percentages = {"car": 20, "bus": 50}
counts = counts_from_percentages(total_vehicles, percentages)

print("Total vehicles:", total_vehicles)
for k, v in counts.items():
    pct = (v / total_vehicles) * 100
    print(f"{k}: {v} vehicles ({pct:.1f}%)")

In [ ]:
import cv2
from ultralytics import YOLO

# --- Settings (edit these) ---
source = "/content/drive/MyDrive/Traffic Dataset/images/test"  # your video file
model_path = "yolov8n.pt"

vehicle_classes = [2, 3, 5, 7]  # COCO: car, motorcycle, bus, truck
conf = 0.25

window_seconds = 5  # how often to decide traffic level
low_threshold = 5     # vehicles/frame (avg) => low traffic
high_threshold = 15   # vehicles/frame (avg) => high traffic

# --- Helper ---
def traffic_level(avg_vehicles: float, low_t: float, high_t: float) -> str:
    if avg_vehicles <= low_t:
        return "low traffic"
    if avg_vehicles >= high_t:
        return "high traffic"
    return "moderate traffic"

# --- Video loop ---
model = YOLO(model_path)
cap = cv2.VideoCapture(source)

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 1:
    fps = 30  # fallback if FPS is not readable

window_frames = max(1, int(fps * window_seconds))

frame_idx = 0
window_total = 0
window_frames_seen = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    # Run detection on this frame (counts all detections for those classes)
    res = model(frame, classes=vehicle_classes, conf=conf, verbose=False)
    boxes = res[0].boxes
    count = 0 if boxes is None else len(boxes)

    window_total += count
    window_frames_seen += 1
    frame_idx += 1

    # Every window -> decide traffic level
    if window_frames_seen >= window_frames:
        avg = window_total / window_frames_seen
        level = traffic_level(avg, low_threshold, high_threshold)
        print(f"Frames up to {frame_idx}: avg vehicles/frame={avg:.2f} => {level}")

        window_total = 0
        window_frames_seen = 0

# Handle leftover frames
if window_frames_seen > 0:
    avg = window_total / window_frames_seen
    level = traffic_level(avg, low_threshold, high_threshold)
    print(f"End: avg vehicles/frame={avg:.2f} => {level}")

cap.release()
